# Time Registry Upload

Uploads **minute-by-minute timestamps** from `2018-01-01` to `2028-01-01` into the `time_registry` table on Supabase. Season is derived from the month using Australian meteorological seasons.

## Imports

In [12]:
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd
import numpy as np
import time

## Supabase Client

In [13]:
load_dotenv()

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

## Load Data

Generates a minute-level datetime range from 2018 to 2028 and maps each record to an Australian meteorological season. `NaN` values are replaced with `None` for Supabase JSON compatibility.

In [20]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    else:
        return "Spring"

dt_range = pd.date_range(start="2018-01-01", end="2028-01-01", freq="min", inclusive="left")

df = pd.DataFrame({
    "datetime_record": dt_range.strftime("%Y-%m-%d %H:%M:%S"),
    "season": [get_season(dt.month) for dt in dt_range]
})
df = df.replace({np.nan: None})
df

,datetime_record,season
0,2018-01-01 00:00:00,Summer
1,2018-01-01 00:01:00,Summer
2,2018-01-01 00:02:00,Summer
3,2018-01-01 00:03:00,Summer
4,2018-01-01 00:04:00,Summer
...,...,...
5258875,2027-12-31 23:55:00,Summer
5258876,2027-12-31 23:56:00,Summer
5258877,2027-12-31 23:57:00,Summer
5258878,2027-12-31 23:58:00,Summer


In [21]:
df.memory_usage(deep=True).apply(lambda x: f"{x / 1024**2:.2f} MB")

Index                0.00 MB
datetime_record    381.16 MB
season             315.96 MB
dtype: str

## Insert

Inserts rows in batches of 1000 with a short sleep between each batch to avoid rate limiting. A running total and elapsed time are printed after each batch.

In [24]:
batch_size = 1000
rows = df.to_dict(orient="records")
total = 1_179_000
start = time.time()

for i in range(1_179_000, len(rows), batch_size):
    batch = rows[i:i + batch_size]
    response = supabase.table("time_registry").upsert(batch, on_conflict="datetime_record", ignore_duplicates=True).execute()
    total += len(response.data)
    elapsed = time.time() - start
    print(f"Inserted {total}/{len(rows)} rows: {elapsed:.1f}s elapsed")
    time.sleep(0.2)

print(f"Finally Done: {time.time() - start:.1f}s total")

Inserted 1179000/5258880 rows: 1.2s elapsed
Inserted 1180000/5258880 rows: 2.0s elapsed
Inserted 1181000/5258880 rows: 2.5s elapsed
Inserted 1182000/5258880 rows: 3.1s elapsed
Inserted 1183000/5258880 rows: 3.7s elapsed
Inserted 1184000/5258880 rows: 4.2s elapsed
Inserted 1185000/5258880 rows: 4.8s elapsed
Inserted 1186000/5258880 rows: 5.4s elapsed
Inserted 1187000/5258880 rows: 5.9s elapsed
Inserted 1188000/5258880 rows: 6.5s elapsed
Inserted 1189000/5258880 rows: 7.0s elapsed
Inserted 1190000/5258880 rows: 7.6s elapsed
Inserted 1191000/5258880 rows: 8.1s elapsed
Inserted 1192000/5258880 rows: 8.6s elapsed
Inserted 1193000/5258880 rows: 9.4s elapsed
Inserted 1194000/5258880 rows: 10.0s elapsed
Inserted 1195000/5258880 rows: 10.5s elapsed
Inserted 1196000/5258880 rows: 11.0s elapsed
Inserted 1197000/5258880 rows: 11.5s elapsed
Inserted 1198000/5258880 rows: 12.1s elapsed
Inserted 1199000/5258880 rows: 12.6s elapsed
Inserted 1200000/5258880 rows: 13.1s elapsed
Inserted 1201000/5258880 